# D3. Cross-Platform Content Adaptation Notebook

> **Related Module**: [D3 Cross-Platform Strategy](../paths/d-platforms/cross-platform-strategy.md)
>
> **Features**: Input an Amazon Listing → One-click generation of Shopify/TikTok/Instagram/Walmart versions
>
> [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kangise/ecommerce-ai-roadmap/blob/main/notebooks/d3-cross-platform-content.ipynb)

In [ ]:
!pip install -q openai pandas

## 1. Input Amazon Listing

In [ ]:
import os, json
from openai import OpenAI
import pandas as pd

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', 'your-key-here')
client = OpenAI(api_key=OPENAI_API_KEY)

# === Replace with your Amazon Listing ===
AMAZON_LISTING = {
    'title': 'Premium Wireless Bluetooth Earbuds - Active Noise Cancellation, 36H Battery Life, IPX5 Waterproof, Deep Bass, Touch Control',
    'bullets': [
        'ACTIVE NOISE CANCELLATION: Advanced ANC blocks background noise. Perfect for commuting and travel.',
        '36-HOUR BATTERY: 8h per charge + 28h with case. Quick charge 10min = 2h playback.',
        'IPX5 WATERPROOF: Sweat and splash resistant for gym and outdoor.',
        'PREMIUM SOUND: 13mm drivers, deep bass, crystal highs. Bluetooth 5.3.',
        'COMFORTABLE: 4 ear tip sizes (XS/S/M/L). Only 5g per earbud.'
    ],
    'price': 39.99,
    'rating': 4.3,
    'review_count': 1250,
    'brand': 'SoundPeak'
}

TARGET_PLATFORMS = ['shopify', 'tiktok_script', 'instagram_caption', 'walmart', 'pinterest_pin']
print(f'Source: Amazon Listing for {AMAZON_LISTING["brand"]}')
print(f'Target platforms: {len(TARGET_PLATFORMS)}')

## 2. AI Cross-Platform Adaptation

In [ ]:
def adapt_to_platform(listing, platform):
    platform_specs = {
        'shopify': 'Shopify product page. Brand-focused, storytelling, longer description, emotional appeal. Include SEO meta title and description.',
        'tiktok_script': 'TikTok 30-second video script. Hook in first 2 seconds (information gap), show problem→solution→CTA. Casual, energetic tone. Include text overlays.',
        'instagram_caption': 'Instagram post caption. 2200 char max. Hook first line, use emojis, include CTA, 20-30 hashtags. Lifestyle-focused.',
        'walmart': 'Walmart Marketplace listing. Shorter title (75 chars), value-focused, price-conscious audience. Emphasize everyday value.',
        'pinterest_pin': 'Pinterest Pin. Title (100 chars) + description (500 chars). SEO-focused, aspirational, include keywords for visual search.'
    }
    
    prompt = f"""Convert this Amazon listing to {platform} format.

Amazon Listing:
Title: {listing['title']}
Bullets: {json.dumps(listing['bullets'])}
Price: ${listing['price']}
Brand: {listing['brand']}

Platform requirements: {platform_specs[platform]}

IMPORTANT: Don't just copy — adapt the tone, length, and style for this specific platform.

Return JSON with platform-appropriate fields."""
    
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': prompt}],
        response_format={'type': 'json_object'},
        max_tokens=800
    )
    return json.loads(response.choices[0].message.content)

results = {}
for platform in TARGET_PLATFORMS:
    print(f'Adapting to {platform}...')
    try:
        results[platform] = adapt_to_platform(AMAZON_LISTING, platform)
        print(f'  ✅ Done')
    except Exception as e:
        print(f'  ❌ Error: {e}')
        results[platform] = {'error': str(e)}

print(f'\n=== {sum(1 for r in results.values() if "error" not in r)}/{len(TARGET_PLATFORMS)} platforms adapted ===')

## 3. Results Display

In [ ]:
platform_emojis = {'shopify': '🛍️', 'tiktok_script': '🎵', 'instagram_caption': '📸', 'walmart': '🏪', 'pinterest_pin': '📌'}

for platform, content in results.items():
    if 'error' in content:
        continue
    emoji = platform_emojis.get(platform, '📄')
    print(f'\n{"="*60}')
    print(f'{emoji} {platform.upper().replace("_", " ")}')
    print(f'{"="*60}')
    for key, value in content.items():
        if isinstance(value, list):
            print(f'\n{key}:')
            for item in value[:5]:
                print(f'  • {item}')
        elif isinstance(value, str) and len(value) > 100:
            print(f'\n{key}:')
            print(f'  {value[:300]}...' if len(value) > 300 else f'  {value}')
        else:
            print(f'{key}: {value}')

## 4. Export

In [ ]:
with open('cross_platform_content.json', 'w') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print('Exported to cross_platform_content.json')

# Simple comparison table
rows = []
for platform, content in results.items():
    if 'error' not in content:
        title = content.get('title', content.get('hook', content.get('pin_title', 'N/A')))
        rows.append({'Platform': platform, 'Title/Hook': str(title)[:80]})
pd.DataFrame(rows).to_csv('platform_titles.csv', index=False)
print('Exported to platform_titles.csv')